## List databases used in a web map

This notebook uses the ArcGIS API for Python to list what databases a web map is pulling from.

In [1]:
from arcgis.gis import GIS
from arcgis.apps.itemgraph import create_dependency_graph

In [2]:
gis=GIS('Pro')
print(gis.users.me.role)

org_admin


### Create a dependency graph for the web map

In [3]:
mapItemId = '59bec7efcc414eba8158a0bba8b3c865'
mapItem = gis.content.get(mapItemId)
graph = create_dependency_graph(gis=gis, item_list=[mapItem])
graph.all_items()

[ItemNode(id: 59bec7efcc414eba8158a0bba8b3c865, item: Electric Utility Map),
 ItemNode(id: 7dc6cea0b1764a1f9af2e679f642f0f5),
 ItemNode(id: 2310fd57cabd4cd1bc3a655959dcf826),
 ItemNode(id: e7bd1ad72a9f4f72b249756d40e60189),
 ItemNode(id: 65f83b9b9ab14fa9abdf87a5ca881ded, item: Electric Utility Map_MIL1),
 ItemNode(id: 08dde22225924129b8a502e8ff46e220)]

### Get the layers in the web map using the ItemGraph

In [4]:
targetNode = graph.get_node(mapItemId)
serviceURLs = {}
for node in targetNode.requires():
    if node.item and node.item.type == "Map Service":
        #serviceURLs.append(node)
        serviceURLs[node.item.url.split('/')[-2]] = node.item.url
        print(f"{node.item.url} added to dictionary")
print(f"{len(serviceURLs)} url(s) collected:")
print(f'\t{serviceURLs}')

https://ebase.ad.local/server/rest/services/Electric_Utility_Map_MIL1/MapServer added to dictionary
1 url(s) collected:
	{'Electric_Utility_Map_MIL1': 'https://ebase.ad.local/server/rest/services/Electric_Utility_Map_MIL1/MapServer'}


### Get server(s) for admin access to services

In [5]:
servers = gis.admin.servers.list()
server = servers[0]

In [6]:
adminServiceURLs = {}
adminServices = server.services.list()
serviceNames = list(serviceURLs.keys())

for s in adminServices:
    sName = s.properties['serviceName']
    if sName in serviceNames:
        print(s.properties['serviceName'])
        adminServiceURLs[sName] = s
        
print(adminServiceURLs)

Electric_Utility_Map_MIL1
{'Electric_Utility_Map_MIL1': <Service at https://ebase.ad.local:6443/arcgis/admin/services/Electric_Utility_Map_MIL1.MapServer>}


In [7]:
for key in serviceNames:
    itemInfo = adminServiceURLs[key].iteminformation
    manifest = itemInfo.manifest
    isReferenced = manifest['databases'][0]['byReference']
    if isReferenced:
        connectionInfo = dict(item.split("=", 1) for item in manifest['databases'][0]['onPremiseConnectionString'].split(';'))
        print(f"Gathering information for: {adminServiceURLs[key]}")
        print(f"\tInstance: {connectionInfo['INSTANCE']},\n\tDatabase: {connectionInfo['DATABASE']},\n\tUser: {connectionInfo['USER']}")

Gathering information for: <Service at https://ebase.ad.local:6443/arcgis/admin/services/Electric_Utility_Map_MIL1.MapServer>
	Instance: sde:postgresql:ebase,
	Database: Electric,
	User: gis
